In [ ]:
import os
import torch
import pandas as pd
import numpy as np
import gcsfs
from google.colab import auth
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from typing import List, Dict

# 1. Authenticate Colab
print("Authenticating with Google Cloud...")
auth.authenticate_user()

# 2. Connect to Bucket
fs = gcsfs.GCSFileSystem()
bucket_path = 'tryanheider-kalshi-bucket/kalshi_dense_state_v1'

print(f"Scanning bucket: gs://{bucket_path}...")
all_dense_files = fs.glob(f"{bucket_path}/*.parquet")
all_dense_files = [f"gs://{f}" for f in all_dense_files]

if not all_dense_files:
    raise FileNotFoundError("No parquet files found in the bucket. Check your path.")

# 3. Pair Markets to Prevent Leakage
market_groups = {}
for file_path in all_dense_files:
    # Example filename: dense_KXMLBGAME-26APR151840KCDET-KC.parquet
    filename = os.path.basename(file_path)
    base_game = filename.split('-')[1] # Extracts '26APR151840KCDET'

    if base_game not in market_groups:
        market_groups[base_game] = []
    market_groups[base_game].append(file_path)

unique_games = list(market_groups.keys())
print(f"Found {len(unique_games)} unique MLB games.")

# Split 80/20 strictly by GAME
train_games, val_games = train_test_split(unique_games, test_size=0.2, random_state=42)

train_files = [f for game in train_games for f in market_groups[game]]
val_files = [f for game in val_games for f in market_groups[game]]

print(f"Train Set: {len(train_games)} games ({len(train_files)} individual tickers)")
print(f"Val Set: {len(val_games)} games ({len(val_files)} individual tickers)")

In [ ]:
class KalshiDenseDataset(Dataset):
    def __init__(self, file_paths: List[str], seq_len: int = 60, forward_look: int = 30, threshold: float = 0.015, stride: int = 5):
        self.seq_len = seq_len
        self.forward_look = forward_look
        self.threshold = threshold
        self.stride = stride

        self.tensors = []
        self.labels = []
        self.timestamps = []
        self.audit_log = {'class_0': 0, 'class_1': 0, 'class_2': 0, 'nan_drops': 0}

        # Explicitly define columns that require log1p normalization
        self.vol_cols = [f'bid_vol_{i}' for i in range(1, 6)] + \
                        [f'ask_vol_{i}' for i in range(1, 6)] + \
                        ['taker_vol_yes', 'taker_vol_no']

        print(f"Loading {len(file_paths)} dense files into RAM...")
        for f in file_paths:
            self._process_file(f)

        print("\n=== FINAL ENGINEERING AUDIT ===")
        print(f"Total valid {seq_len}-step sequences: {len(self.tensors)}")
        print(f"Distribution -> DOWN: {self.audit_log['class_0']} | FLAT: {self.audit_log['class_1']} | UP: {self.audit_log['class_2']}")
        print(f"Dropped due to NaNs: {self.audit_log['nan_drops']}")

    def _process_file(self, file_path: str):
        df = pd.read_parquet(file_path)
        if df.empty:
            return

        # 1. Target Math (Must be done BEFORE log normalization)
        # Adding 1e-8 prevents division by zero if book momentarily empties
        df['micro_price'] = (df['ask_vol_1'] * df['bid_px_1'] + df['bid_vol_1'] * df['ask_px_1']) / (df['bid_vol_1'] + df['ask_vol_1'] + 1e-8)
        df['smoothed_target'] = df['micro_price'].rolling(10).mean().shift(-self.forward_look)

        # 2. Log1p Normalization
        for col in self.vol_cols:
            df[col] = np.log1p(df[col].values)

        # 3. Define Feature Vector (Drop target metrics and pure ts)
        feature_cols = [c for c in df.columns if c not in ['ts', 'micro_price', 'smoothed_target']]

        # 4. Extract Sliding Windows
        for i in range(0, len(df) - self.seq_len - self.forward_look, self.stride):
            window = df.iloc[i : i + self.seq_len][feature_cols].values

            # Labeling is strictly based on the final step of the window vs the future
            current_micro = df.iloc[i + self.seq_len - 1]['micro_price']
            future_micro = df.iloc[i + self.seq_len - 1]['smoothed_target']
            target_px_move = future_micro - current_micro

            trigger_ts = df.iloc[i + self.seq_len - 1]['ts']

            if np.isnan(window).any() or np.isnan(target_px_move):
                self.audit_log['nan_drops'] += 1
                continue

            # Classify: 0 (Down), 1 (Flat), 2 (Up)
            if target_px_move > self.threshold:
                label = 2
                self.audit_log['class_2'] += 1
            elif target_px_move < -self.threshold:
                label = 0
                self.audit_log['class_0'] += 1
            else:
                label = 1
                self.audit_log['class_1'] += 1

            self.tensors.append(torch.tensor(window, dtype=torch.float32))
            self.labels.append(label)
            self.timestamps.append(trigger_ts)

    def __len__(self):
        return len(self.tensors)

    def __getitem__(self, idx):
        return self.tensors[idx], self.labels[idx], self.timestamps[idx]

# Instantiate DataLoaders
print("\nBuilding Training Dataset...")
train_dataset = KalshiDenseDataset(train_files, seq_len=60, forward_look=30, threshold=0.015, stride=5)

print("\nBuilding Validation Dataset...")
val_dataset = KalshiDenseDataset(val_files, seq_len=60, forward_look=30, threshold=0.015, stride=5)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

print(f"\nDataLoader Ready. Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        pe: Tensor = torch.zeros(max_len, d_model)
        position: Tensor = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term: Tensor = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        # Shape: (1, max_len, d_model) for batch_first=True
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x: Tensor) -> Tensor:
        """
        Args:
            x: Tensor, shape [batch_size, seq_len, embedding_dim]
        """
        x = x + self.pe[:, :x.size(1), :]
        return x

class KalshiTransformer(nn.Module):
    def __init__(self, feature_dim: int = 23, d_model: int = 64, nhead: int = 4, num_layers: int = 2, num_classes: int = 3):
        super().__init__()
        self.d_model: int = d_model

        # 1. Input Projection: Map 23 raw features to 64-dim embedding
        self.input_proj = nn.Linear(feature_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model)

        # 2. Transformer Encoder (batch_first=True ensures shape is [batch, seq, feature])
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True, dropout=0.1)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # 3. Classification Head
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(32, num_classes)
        )

    def forward(self, x: Tensor) -> Tensor:
        """
        Args:
            x: Tensor, shape [batch_size, seq_len, feature_dim]
        Returns:
            logits: Tensor, shape [batch_size, num_classes]
        """
        # Project and encode
        x = self.input_proj(x) * math.sqrt(self.d_model)
        x = self.pos_encoder(x)

        # Pass through attention layers
        x = self.transformer_encoder(x)

        # Extract the final time-step representation for sequence classification
        final_step: Tensor = x[:, -1, :]

        logits: Tensor = self.classifier(final_step)
        return logits



class FocalLoss(nn.Module):
    def __init__(self, alpha: Tensor, gamma: float = 2.0):
        super().__init__()
        self.alpha: Tensor = alpha
        self.gamma: float = gamma

    def forward(self, inputs: Tensor, targets: Tensor) -> Tensor:
        """
        Args:
            inputs: [batch_size, num_classes] (raw logits)
            targets: [batch_size] (integer class indices)
        """
        ce_loss: Tensor = F.cross_entropy(inputs, targets, reduction='none')
        pt: Tensor = torch.exp(-ce_loss)

        # Gather the alpha weights for the target classes
        alpha_t: Tensor = self.alpha.gather(0, targets)

        # Apply the focal equation
        focal_loss: Tensor = alpha_t * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

In [ ]:
from tqdm import tqdm

# --- Setup Devices and Hyperparameters ---
device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model: nn.Module = KalshiTransformer(feature_dim=23, d_model=64, nhead=4, num_layers=2).to(device)
optimizer: torch.optim.Optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

# Calculate empirical Alpha weights based on your Train distribution
# Inverse frequency weighting: total_samples / (num_classes * class_samples)
class_counts = torch.tensor([5084, 87474, 4750], dtype=torch.float32)
alpha_weights: Tensor = class_counts.sum() / (3.0 * class_counts)
alpha_weights = alpha_weights.to(device)

criterion: nn.Module = FocalLoss(alpha=alpha_weights, gamma=2.0)

# --- Training Loop ---
epochs: int = 10
best_val_loss: float = float('inf')

for epoch in range(epochs):
    model.train()
    running_loss: float = 0.0
    correct: int = 0
    total: int = 0

    # Progress bar wrapping the training loader
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")

    for X_batch, Y_batch, _ in train_bar:
        X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)

        optimizer.zero_grad()
        logits: Tensor = model(X_batch)
        loss: Tensor = criterion(logits, Y_batch)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) # Prevent explosions
        optimizer.step()

        running_loss += loss.item()

        # Compute basic accuracy
        preds: Tensor = torch.argmax(logits, dim=1)
        correct += (preds == Y_batch).sum().item()
        total += Y_batch.size(0)

        # Update progress bar metrics
        train_bar.set_postfix({'Loss': f"{loss.item():.4f}"})

    train_acc: float = correct / total

    # --- Validation ---
    model.eval()
    val_loss: float = 0.0
    val_correct: int = 0
    val_total: int = 0

    with torch.no_grad():
        val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val  ]")
        for X_batch, Y_batch, _ in val_bar:
            X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)

            logits: Tensor = model(X_batch)
            loss: Tensor = criterion(logits, Y_batch)

            val_loss += loss.item()
            preds: Tensor = torch.argmax(logits, dim=1)
            val_correct += (preds == Y_batch).sum().item()
            val_total += Y_batch.size(0)

    val_acc: float = val_correct / val_total


    print(f"Epoch {epoch+1} Summary -> Train Loss: {running_loss/len(train_loader):.4f} | Train Acc: {train_acc:.4f} || Val Loss: {val_loss/len(val_loader):.4f} | Val Acc: {val_acc:.4f}\n")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        # Save the physical weights to your VM
        torch.save(model.state_dict(), 'best_alpha_model.pth')
        print(f"  [*] Val Loss improved. Saved best_alpha_model.pth")

In [ ]:
import os
import glob
import tarfile

# --- CONFIGURATION ---
# Pointing directly to the root of the bucket for the raw vaults
RAW_BUCKET_URI = "gs://tryanheider-kalshi-bucket/raw_vault_*.tar.gz"
DENSE_BUCKET_URI = "gs://tryanheider-kalshi-bucket/kalshi_dense_state_v1/*.parquet"

LOCAL_ARCHIVE_DIR = "/content/kalshi_raw_archives"
LOCAL_EXTRACT_DIR = "/content/kalshi_extracted"
LOCAL_DENSE_DIR = "/content/kalshi_dense"

os.makedirs(LOCAL_ARCHIVE_DIR, exist_ok=True)
os.makedirs(LOCAL_EXTRACT_DIR, exist_ok=True)
os.makedirs(LOCAL_DENSE_DIR, exist_ok=True)

print("=== BOOTING DATA FEEDER ===")

print("1. Syncing raw archives and dense tensors from Google Cloud...")
# gcloud storage cp handles wildcards natively. 2>/dev/null hides Colab's cosmetic warnings.
!gcloud storage cp {RAW_BUCKET_URI} {LOCAL_ARCHIVE_DIR}/ 2>/dev/null
!gcloud storage cp {DENSE_BUCKET_URI} {LOCAL_DENSE_DIR}/ 2>/dev/null

print("\n2. Extracting raw asynchronous event logs via pure Python...")
archives = glob.glob(f"{LOCAL_ARCHIVE_DIR}/*.tar*")

if not archives:
    print("CRITICAL ERROR: No archives found to extract! Download failed.")
else:
    for archive_path in archives:
        base_name = os.path.basename(archive_path).replace('.tar.gz', '').replace('.tar', '')
        target_dir = os.path.join(LOCAL_EXTRACT_DIR, base_name)

        if not os.path.exists(target_dir) or not os.listdir(target_dir):
            print(f"Unpacking {base_name}...")
            os.makedirs(target_dir, exist_ok=True)
            with tarfile.open(archive_path, 'r') as tar:
                # Python 3.14 filter warning is harmless here, but we can explicitly trust the archive
                tar.extractall(path=target_dir)
            print(f"  -> {base_name} extracted successfully.")
        else:
            print(f"Archive {base_name} already unpacked. Skipping.")

# Verification Check
raw_parquets = glob.glob(f"{LOCAL_EXTRACT_DIR}/**/*.parquet", recursive=True)
dense_parquets = glob.glob(f"{LOCAL_DENSE_DIR}/*.parquet")

print("\n=== FEEDER COMPLETE ===")
print(f"Raw Parquet Files Secured: {len(raw_parquets)}")
print(f"Dense Tensor Files Secured: {len(dense_parquets)}")

In [ ]:
from dataclasses import dataclass
from typing import Optional, List
import pandas as pd
import numpy as np

@dataclass
class TradeReceipt:
    """The master audit log for every signal the model fires."""
    # Contextual Intel
    signal_time: float
    confidence: float

    # Microstructure Intel
    intended_side: str          # 'yes' or 'no'
    entry_price: float
    spread_width: float
    queue_ahead: float

    # Execution Outcome
    entry_status: str           # 'FILLED', 'CANCELLED_TTL'
    exit_status: Optional[str]  # 'SPREAD_CAPTURED', 'FORCED_MARKET_OUT', None

    # Sizing & PnL
    total_taker_flow_seen: float = 0.0  # Used to calculate max potential position size
    realized_pnl_cents: float = 0.0

from typing import Optional
import numpy as np

class ProductionRouter:
    """
    This block perfectly mirrors your live trading bot's logic.
    It evaluates signals based purely on the 4-hour ML window.
    """
    def __init__(self, conf_threshold: float = 0.75, max_spread_cents: float = 0.05):
        self.conf_threshold = conf_threshold
        self.max_spread = max_spread_cents

    def evaluate_signal(self, pred_class: int, confidence: float, current_state: np.ndarray, timestamp: float) -> Optional[dict]:
        """
        Evaluates the tensor state against strict production risk rules.
        """
        # 1. Base Alpha Check
        if pred_class == 1 or confidence < self.conf_threshold:
            return None

        # State mapping (0=bid_px, 1=bid_vol, 2=ask_px, 3=ask_vol)
        bid_px = current_state[0]
        ask_px = current_state[2]
        spread = round(ask_px - bid_px, 2)

        # 2. Microstructure Risk Check (Don't trade into a blown-out spread)
        if spread > self.max_spread:
            return None

        # 3. Generate the Order
        if pred_class == 0: # DOWN -> We want to sell YES (Join Ask)
            side_to_join = 'no'
            target_price = ask_px
            queue_ahead = np.expm1(current_state[3])
        else:               # UP -> We want to buy YES (Join Bid)
            side_to_join = 'yes'
            target_price = bid_px
            queue_ahead = np.expm1(current_state[1])

        return {
            'side': side_to_join,
            'price': target_price,
            'queue_ahead': queue_ahead,
            'spread_width': spread,
            'confidence': confidence,
            'signal_ts': timestamp
        }

In [ ]:
import pandas as pd
import numpy as np
from typing import Optional

class ExchangeEmulator:
    def __init__(self, latency_penalty_ms: int = 50, entry_ttl: int = 15, exit_ttl: int = 60, assumed_exit_queue: float = 200.0):
        self.latency = latency_penalty_ms / 1000.0
        self.entry_ttl = entry_ttl
        self.exit_ttl = exit_ttl
        self.assumed_exit_queue = assumed_exit_queue # Proxy for how deep the queue is when we post our exit

    def process_order(self, order: dict, market_df: pd.DataFrame, all_timestamps: np.ndarray, all_states: np.ndarray) -> TradeReceipt:
        arrival_time = order['signal_ts'] + self.latency

        # Initialize the Audit Receipt
        receipt = TradeReceipt(
            signal_time=order['signal_ts'],
            confidence=order['confidence'],
            intended_side=order['side'],
            entry_price=order['price'],
            spread_width=order['spread_width'],
            queue_ahead=order['queue_ahead'],
            entry_status='PENDING',
            exit_status=None,
            total_taker_flow_seen=0.0,
            realized_pnl_cents=0.0
        )

        # 1. ENTRY LEG
        future_df = market_df[market_df['ts'] >= arrival_time]
        if future_df.empty:
            receipt.entry_status = 'CANCELLED_MARKET_CLOSED'
            return receipt

        queue_ahead = order['queue_ahead']
        fill_time = None

        for _, event in future_df.iterrows():
            current_time = event['ts']

            if current_time - arrival_time > self.entry_ttl:
                receipt.entry_status = 'CANCELLED_TTL'
                return receipt

            if event['event_type'] == 'trade':
                yes_px = event.get('yes_price_dollars')
                if yes_px is None or pd.isna(yes_px): continue

                trade_price = float(yes_px) if order['side'] == 'yes' else round(1.0 - float(yes_px), 2)
                trade_side = event.get('taker_side', 'yes')

                # Opposing taker flow sweeps our price level
                if trade_price == order['price'] and trade_side != order['side']:
                    trade_vol = float(event.get('count_fp', event.get('count', 0.0)))
                    receipt.total_taker_flow_seen += trade_vol
                    queue_ahead -= trade_vol

                    if queue_ahead <= 0 and fill_time is None:
                        receipt.entry_status = 'FILLED'
                        fill_time = current_time
                        # We don't break here! We must enter the EXIT LEG loop immediately.
                        break

        # If we never got filled in the entry loop
        if receipt.entry_status != 'FILLED':
            receipt.entry_status = 'CANCELLED_ADVERSE_MOVE'
            return receipt

        # 2. EXIT LEG (We hold inventory)
        # We target capturing 1 cent.
        # If we bought YES (side='yes'), we want to sell at entry + 0.01
        # If we sold YES (side='no'), we want to buy back at entry - 0.01
        exit_price = round(order['price'] + 0.01, 2) if order['side'] == 'yes' else round(order['price'] - 0.01, 2)
        exit_queue_ahead = self.assumed_exit_queue

        exit_reality = market_df[market_df['local_recv_ts'] >= fill_time]

        for _, event in exit_reality.iterrows():
            current_time = event['ts']

            # EXIT TTL EXPIRED: PANIC CROSS THE SPREAD
            if current_time - fill_time > self.exit_ttl:
                receipt.exit_status = 'FORCED_MARKET_OUT'
                receipt.realized_pnl_cents = self._calculate_market_out_pnl(
                    current_time, order['side'], order['price'], all_timestamps, all_states
                )
                return receipt

            if event['event_type'] == 'trade':
                yes_px = event.get('yes_price_dollars')
                if yes_px is None or pd.isna(yes_px): continue

                trade_price = float(yes_px) if order['side'] == 'yes' else round(1.0 - float(yes_px), 2)
                trade_side = event.get('taker_side', 'yes')

                # Taker flow hits our exit order
                if trade_price == exit_price and trade_side == order['side']:
                    trade_vol = float(event.get('count_fp', event.get('count', 0.0)))
                    exit_queue_ahead -= trade_vol

                    if exit_queue_ahead <= 0:
                        receipt.exit_status = 'SPREAD_CAPTURED'
                        receipt.realized_pnl_cents = 0.01 #FIX TO CENTS *100 LATER
                        return receipt

        # Market closed while holding bag
        receipt.exit_status = 'FORCED_MARKET_OUT'
        receipt.realized_pnl_cents = -0.05 # Default severe penalty if data completely cuts off
        return receipt

    def _calculate_market_out_pnl(self, panic_time: float, intended_side: str, entry_price: float, all_timestamps: np.ndarray, all_states: np.ndarray) -> float:
        """Finds the exact resting bid/ask at the moment of panic and crosses it."""
        # Find the closest tensor timestamp to our panic_time
        idx = np.searchsorted(all_timestamps, panic_time)
        if idx >= len(all_timestamps):
            idx = len(all_timestamps) - 1

        panic_state = all_states[idx][-1]

        if intended_side == 'yes':
            # We are LONG YES. We must SELL to the Best Bid.
            best_bid = panic_state[0] # bid_px_1
            return round(best_bid - entry_price, 2)
        else:
            # We are SHORT YES. We must BUY from the Best Ask.
            best_ask = panic_state[2] # ask_px_1
            return round(entry_price - best_ask, 2)

In [ ]:
import os
import glob
import torch
import pandas as pd
import numpy as np
import torch.nn.functional as F
from torch.utils.data import DataLoader

#BUGGED? def run_master_backtest(alpha_model: torch.nn.Module, val_dense_files: list, raw_extracted_dir: str = './kalshi_extracted'):
    """
    The Master Evaluation Loop.
    Stitches together the Feeder, Alpha Engine, Router, and Emulator.
    """
    print(f"=== INITIATING MASTER BACKTEST ACROSS {len(val_dense_files)} MARKETS ===")

    alpha_model.eval()

    # 1. Instantiate the execution layer blocks
    router = ProductionRouter(conf_threshold=0.75, max_spread_cents=0.05)
    emulator = ExchangeEmulator(latency_penalty_ms=50, entry_ttl=15, exit_ttl=60)

    all_trade_receipts = []

    for dense_file in val_dense_files:
        ticker_name = os.path.basename(dense_file).replace('dense_', '').replace('.parquet', '')

        # --- BLOCK 1: THE FEEDER ---
        search_path = f"{raw_extracted_dir}/**/{ticker_name}/*.parquet"
        raw_files = glob.glob(search_path, recursive=True)
        if not raw_files:
            print(f"[{ticker_name}] Raw data missing. Skipping.")
            continue

        raw_df = pd.concat([pd.read_parquet(f) for f in raw_files], ignore_index=True)
        raw_df = raw_df.sort_values(by=['local_recv_ts', 'seq']).reset_index(drop=True)

        # Load Dense Tensors
        dataset = KalshiDenseDataset([dense_file], window_size=60, forward_look=30, threshold=0.015, stride=1)
        if len(dataset) == 0:
            continue
        loader = DataLoader(dataset, batch_size=256, shuffle=False)

        # --- BLOCK 2: THE ALPHA ENGINE ---
        preds, probs, states, timestamps = [], [], [], []
        with torch.no_grad():
            # Unpacking all 3 items (including timestamps)
            for X_batch, _, ts_batch in loader:
                X_batch = X_batch.to(device)
                logits = alpha_model(X_batch)
                batch_probs = F.softmax(logits, dim=1)

                preds.append(torch.argmax(batch_probs, dim=1).cpu().numpy())
                probs.append(batch_probs.cpu().numpy())
                states.append(X_batch.cpu().numpy())
                timestamps.append(ts_batch.numpy())

        preds = np.concatenate(preds, axis=0)
        probs = np.concatenate(probs, axis=0)
        states = np.concatenate(states, axis=0)
        timestamps = np.concatenate(timestamps, axis=0)


        # --- BLOCK 3 & 4: ROUTING AND EXECUTION ---
        for i in range(len(preds)):
            # Route the signal
            order = router.evaluate_signal(
                pred_class=preds[i],
                confidence=probs[i, preds[i]],
                current_state=states[i][-1],
                timestamp=timestamps[i]
            )

            # If the router approves the trade, send it to the Exchange Emulator
            if order is not None:
                receipt = emulator.process_order(
                    order=order,
                    market_df=raw_df,
                    all_timestamps=timestamps,
                    all_states=states
                )
                all_trade_receipts.append(receipt)

    # Compile the final Intel
    return generate_audit_report(all_trade_receipts)

def generate_audit_report(receipts: list) -> pd.DataFrame:
    """Converts TradeReceipt objects into a Pandas DataFrame for deep analysis."""
    if not receipts:
        print("Zero trades were executed.")
        return pd.DataFrame()

    df_audit = pd.DataFrame([vars(r) for r in receipts])

    total_signals = len(df_audit)
    filled_entries = len(df_audit[df_audit['entry_status'] == 'FILLED'])
    spreads_captured = len(df_audit[df_audit['exit_status'] == 'SPREAD_CAPTURED'])
    market_outs = len(df_audit[df_audit['exit_status'] == 'FORCED_MARKET_OUT'])
    total_pnl = df_audit['realized_pnl_cents'].sum()

    print("\n=== MASTER BACKTEST AUTOPSY ===")
    print(f"Signals Fired by Router: {total_signals}")
    print(f"Entries Filled by Exchange: {filled_entries}")
    print(f"Entries Cancelled/Missed: {total_signals - filled_entries}")
    print("-" * 35)
    print(f"Inventory Outcomes:")
    print(f"  [WIN] Spreads Captured: {spreads_captured}")
    print(f"  [LOSS] Forced Market-Outs: {market_outs}")
    print("=" * 35)
    # Divided by 100 to convert pennies to actual dollars
    print(f"TRUE REALIZED PNL: ${total_pnl / 100:.2f}")

    return df_audit

In [ ]:
# 1. Initialize a fresh Transformer matching the EXACT trained architecture
production_model: torch.nn.Module = KalshiTransformer(
    feature_dim=23,
    d_model=64,
    nhead=4,
    num_layers=2
).to(device)

# 2. Load the peak weights
production_model.load_state_dict(torch.load('best_alpha_model.pth', weights_only=True))
production_model.eval()

# 3. Run the Master Backtest
df_audit: pd.DataFrame = run_master_backtest(
    alpha_model=production_model,
    val_dense_files=val_files,
    raw_extracted_dir='./kalshi_extracted'
)

In [ ]:
import os
import glob
import tarfile

BUCKET_URI = "gs://tryanheider-kalshi-bucket/raw_vault_*.tar.gz"
LOCAL_ARCHIVE_DIR = "/content/kalshi_raw_archives"
LOCAL_EXTRACT_DIR = "/content/kalshi_extracted"

os.makedirs(LOCAL_ARCHIVE_DIR, exist_ok=True)
os.makedirs(LOCAL_EXTRACT_DIR, exist_ok=True)

print("1. Syncing archives from the bucket (Warnings are muted, please wait...)")
# The 2>/dev/null hides Colab's cosmetic component warnings
!gcloud storage cp {BUCKET_URI} {LOCAL_ARCHIVE_DIR}/ 2>/dev/null

print("\n2. Extracting via pure Python (Safe Mode)...")
archives = glob.glob(f"{LOCAL_ARCHIVE_DIR}/*.tar*")

if not archives:
    print("CRITICAL ERROR: No archives found to extract! Download failed.")
else:
    for archive_path in archives:
        base_name = os.path.basename(archive_path).replace('.tar.gz', '').replace('.tar', '')
        target_dir = os.path.join(LOCAL_EXTRACT_DIR, base_name)

        if not os.path.exists(target_dir) or not os.listdir(target_dir):
            print(f"Unpacking {base_name}...")
            os.makedirs(target_dir, exist_ok=True)
            with tarfile.open(archive_path, 'r') as tar:
                tar.extractall(path=target_dir)
            print(f"  -> {base_name} extracted successfully.")
        else:
            print(f"Archive {base_name} already unpacked.")

print("\n3. VERIFICATION (Looking for physical .parquet files):")
parquet_files = glob.glob(f"{LOCAL_EXTRACT_DIR}/**/*.parquet", recursive=True)
print(f"TOTAL PARQUET FILES FOUND: {len(parquet_files)}")

if len(parquet_files) > 0:
    print("\nSAMPLE PATH FOUND:")
    print(parquet_files[0])

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

def generate_audit_report(receipts: list) -> pd.DataFrame:
    """Converts TradeReceipt objects into a Pandas DataFrame for deep analysis."""
    if not receipts:
        print("Zero trades were executed.")
        return pd.DataFrame()

    df_audit = pd.DataFrame([vars(r) for r in receipts])

    total_signals = len(df_audit)
    filled_entries = len(df_audit[df_audit['entry_status'] == 'FILLED'])
    spreads_captured = len(df_audit[df_audit['exit_status'] == 'SPREAD_CAPTURED'])
    market_outs = len(df_audit[df_audit['exit_status'] == 'FORCED_MARKET_OUT'])
    total_pnl = df_audit['realized_pnl_cents'].sum()

    print("\n=== MASTER BACKTEST AUTOPSY ===")
    print(f"Signals Fired by Router: {total_signals}")
    print(f"Entries Filled by Exchange: {filled_entries}")
    print(f"Entries Cancelled/Missed: {total_signals - filled_entries}")
    print("-" * 35)
    print(f"Inventory Outcomes:")
    print(f"  [WIN] Spreads Captured: {spreads_captured}")
    print(f"  [LOSS] Forced Market-Outs: {market_outs}")
    print("=" * 35)
    # Divided by 100 to convert pennies to actual dollars
    print(f"TRUE REALIZED PNL: ${total_pnl / 100:.2f}")

    return df_audit

def run_master_backtest(alpha_model: torch.nn.Module, val_dense_files: list, raw_extracted_dir: str = '/content/kalshi_extracted'):
    """The Master Evaluation Loop with bulletproof os.walk data hunting."""
    print(f"=== INITIATING MASTER BACKTEST ACROSS {len(val_dense_files)} MARKETS ===")

    alpha_model.eval()

    router = ProductionRouter(conf_threshold=0.6, max_spread_cents=0.05)
    emulator = ExchangeEmulator(latency_penalty_ms=50, entry_ttl=15, exit_ttl=60)

    all_trade_receipts = []

    for dense_file in val_dense_files:
        ticker_name = os.path.basename(dense_file).replace('dense_', '').replace('.parquet', '')

        # --- BLOCK 1: THE FEEDER (Bulletproof Pathing) ---
        raw_files = []
        for root, dirs, files in os.walk(raw_extracted_dir):
            if ticker_name in root:
                for file in files:
                    if file.endswith('.parquet'):
                        raw_files.append(os.path.join(root, file))

        if not raw_files:
            print(f"[{ticker_name}] Raw data missing. Skipping.")
            continue

        raw_df: pd.DataFrame = pd.concat([pd.read_parquet(f) for f in raw_files], ignore_index=True)

        # --- NEW: Mixed-Schema Timestamp Parsing ---
        # 1. Try to cast everything to pure float (catches strings like '1776358576')
        parsed_ts: pd.Series = pd.to_numeric(raw_df['ts'], errors='coerce')

        # 2. Find anything that failed (the ISO strings) and parse them using datetime
        iso_mask: pd.Series = parsed_ts.isna()
        if iso_mask.any():
            parsed_ts[iso_mask] = pd.to_datetime(raw_df.loc[iso_mask, 'ts'], format='mixed', utc=True).astype('int64') / 1e9

        raw_df['ts'] = parsed_ts
        raw_df = raw_df.sort_values(by=['ts', 'seq']).reset_index(drop=True)

        # --- BLOCK 2: LOAD TENSORS ---
        dataset = KalshiDenseDataset([dense_file], seq_len=60, forward_look=30, threshold=0.015, stride=1)
        if len(dataset) == 0:
            continue
        loader = DataLoader(dataset, batch_size=256, shuffle=False)

        # --- BLOCK 3: THE ALPHA ENGINE ---
        preds, probs, states, timestamps = [], [], [], []
        with torch.no_grad():
            for X_batch, _, ts_batch in loader:
                X_batch = X_batch.to(device)
                logits = alpha_model(X_batch)
                batch_probs = F.softmax(logits, dim=1)

                preds.append(torch.argmax(batch_probs, dim=1).cpu().numpy())
                probs.append(batch_probs.cpu().numpy())
                states.append(X_batch.cpu().numpy())
                timestamps.append(ts_batch.numpy())

        preds = np.concatenate(preds, axis=0)
        probs = np.concatenate(probs, axis=0)
        states = np.concatenate(states, axis=0)
        timestamps = np.concatenate(timestamps, axis=0)

        # --- BLOCK 4: ROUTING AND EXECUTION ---
        for i in range(len(preds)):
            order = router.evaluate_signal(
                pred_class=preds[i],
                confidence=probs[i, preds[i]],
                current_state=states[i][-1],
                timestamp=timestamps[i]
            )

            if order is not None:
                receipt = emulator.process_order(
                    order=order,
                    market_df=raw_df,
                    all_timestamps=timestamps,
                    all_states=states
                )
                all_trade_receipts.append(receipt)

    return generate_audit_report(all_trade_receipts)


# ==========================================
# === 🚀 EXECUTE THE BACKTEST SIMULATION ===
# ==========================================

# 1. Initialize the Transformer matching your exact trained architecture
production_model = KalshiTransformer(
    feature_dim=23,
    d_model=64,
    nhead=4,
    num_layers=2
).to(device)

# 2. Load the peak weights saved during your training run
production_model.load_state_dict(torch.load('best_alpha_model.pth', weights_only=True))
production_model.eval()

# 3. Run the Master Backtest
df_audit = run_master_backtest(
    alpha_model=production_model,
    val_dense_files=val_files,
    raw_extracted_dir='/content/kalshi_extracted'
)

# 4. Instantly hunt for intel
if not df_audit.empty:
    print("\n--- Intel: 1-Cent Spread Trades ---")
    print(df_audit[df_audit['spread_width'] == 0.01].head())

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix
from typing import List
import torch
from torch.utils.data import DataLoader

def analyze_thresholds(alpha_model: torch.nn.Module, val_loader: DataLoader, thresholds: List[float]) -> None:
    """
    Evaluates hit rates and confusion matrices across confidence thresholds
    without running the heavy exchange execution simulator.
    """
    alpha_model.eval()
    all_probs_list: List[np.ndarray] = []
    all_targets_list: List[np.ndarray] = []

    with torch.no_grad():
        for X_batch, Y_batch, _ in val_loader:
            logits: torch.Tensor = alpha_model(X_batch.to(device))
            probs: torch.Tensor = torch.nn.functional.softmax(logits, dim=1)

            all_probs_list.append(probs.cpu().numpy())
            all_targets_list.append(Y_batch.numpy())

    # p_matrix shape: [N, 3] representing probabilities for [DOWN, FLAT, UP]
    p_matrix: np.ndarray = np.concatenate(all_probs_list, axis=0)
    y_true: np.ndarray = np.concatenate(all_targets_list, axis=0)

    print("=== MODEL CONFIDENCE AUTOSPY ===")

    for t in thresholds:
        max_probs: np.ndarray = np.max(p_matrix, axis=1)
        valid_idx: np.ndarray = np.where(max_probs >= t)[0]

        if len(valid_idx) == 0:
            print(f"\n[Threshold {t:.2f}] 0 signals fired.")
            continue

        y_pred_t: np.ndarray = np.argmax(p_matrix[valid_idx], axis=1)
        y_true_t: np.ndarray = y_true[valid_idx]

        hit_rate: float = np.mean(y_pred_t == y_true_t)
        cm: np.ndarray = confusion_matrix(y_true_t, y_pred_t, labels=[0, 1, 2])

        print(f"\n[Threshold {t:.2f}] Total Signals: {len(valid_idx)} | Hit Rate: {hit_rate:.2%}")
        print(f"Confusion Matrix (Rows: True, Cols: Pred [DOWN, FLAT, UP]):")
        print(cm)

# --- Execution ---
# Assumes 'loader' is your standard validation DataLoader containing the 14,000 sequences
analyze_thresholds(production_model, val_loader, thresholds=[0.50, 0.55, 0.60, 0.65, 0.70, 0.75])

In [ ]:
import pandas as pd
from typing import Dict

def analyze_existing_audit(df: pd.DataFrame) -> None:
    """Dissects the existing TradeReceipt DataFrame without re-running the simulation."""
    if df.empty:
        print("DataFrame is empty.")
        return

    print("=== DEEP DIVE: ENTRY DYNAMICS ===")
    entry_stats = df.groupby('entry_status').agg(
        Count=('signal_time', 'count'),
        Avg_Queue_Ahead=('queue_ahead', 'mean'),
        Avg_Spread=('spread_width', 'mean')
    ).round(2)
    print(entry_stats)

    print("\n=== DEEP DIVE: INVENTORY (FILLED TRADES) ===")
    filled_df: pd.DataFrame = df[df['entry_status'] == 'FILLED']

    if filled_df.empty:
        print("No filled trades to analyze.")
        return

    exit_stats = filled_df.groupby(['intended_side', 'exit_status']).agg(
        Count=('signal_time', 'count'),
        Total_PnL_Cents=('realized_pnl_cents', 'sum'),
        Avg_PnL_Cents=('realized_pnl_cents', 'mean')
    ).round(3)
    print(exit_stats)

    print("\n=== WORST 5 ADVERSE SELECTION LOSSES ===")
    worst_losses = filled_df.sort_values('realized_pnl_cents').head(5)
    print(worst_losses[['intended_side', 'entry_price', 'exit_status', 'realized_pnl_cents', 'queue_ahead']])

# Run this instantly on your existing data
analyze_existing_audit(df_audit)

In [ ]:
import pandas as pd
from datetime import datetime
import pytz
from typing import List

def audit_specific_trade(df_audit: pd.DataFrame, dense_files: List[str], target_loss: float = -0.72) -> None:
    """Isolates the specific trade, converts the timestamp, and hunts down the ticker."""
    trade: pd.DataFrame = df_audit[df_audit['realized_pnl_cents'] == target_loss]

    if trade.empty:
        print("Trade not found.")
        return

    unix_ts: float = trade['signal_time'].iloc[0]

    # Convert to Eastern Time (Standard for MLB tracking)
    est_tz = pytz.timezone('US/Eastern')
    human_time: str = datetime.fromtimestamp(unix_ts, est_tz).strftime('%Y-%m-%d %I:%M:%S %p %Z')

    print("=== THE RECEIPT ===")
    print(f"Unix Timestamp: {unix_ts}")
    print(f"Wall-Clock Time: {human_time}")
    print(f"Entry Price: ${trade['entry_price'].iloc[0]}")
    print("-" * 30)

    print("Hunting for the exact market ticker...")
    found_ticker: str = "Unknown"
    for file in dense_files:
        # Load just the timestamps to keep RAM usage low
        df: pd.DataFrame = pd.read_parquet(file, columns=['local_recv_ts'])
        if not df.empty and df['local_recv_ts'].min() <= unix_ts <= df['local_recv_ts'].max():
            found_ticker = file.split('/')[-1].replace('dense_', '').replace('.parquet', '')
            break

    print(f"Market Ticker: {found_ticker}")

# Run this instantly
audit_specific_trade(df_audit, val_files)

In [ ]:
import pandas as pd
from datetime import datetime
import pytz

def audit_worst_trade(df_audit: pd.DataFrame, dense_files: list) -> None:
    """Isolates the absolute worst trade by sorting, bypassing float precision bugs."""
    if df_audit.empty or 'FILLED' not in df_audit['entry_status'].values:
        print("No filled trades to audit.")
        return

    # Isolate filled trades and sort by realized PnL ascending (worst losses first)
    filled_df = df_audit[df_audit['entry_status'] == 'FILLED']
    worst_trade = filled_df.sort_values('realized_pnl_cents').iloc[0]

    unix_ts = worst_trade['signal_time']
    exact_loss = worst_trade['realized_pnl_cents']

    # Convert to Eastern Time
    est_tz = pytz.timezone('US/Eastern')
    human_time = datetime.fromtimestamp(unix_ts, est_tz).strftime('%Y-%m-%d %I:%M:%S %p %Z')

    print("=== THE RECEIPT ===")
    print(f"Unix Timestamp: {unix_ts}")
    print(f"Wall-Clock Time: {human_time}")
    print(f"Entry Price: ${worst_trade['entry_price']}")
    print(f"Exact Loss: ${exact_loss}")
    print("-" * 30)

    print("Hunting for the exact market ticker...")
    found_ticker = "Unknown"
    print("Hunting for the exact market ticker...")
    found_ticker = "Unknown"
    for file in dense_files:
        df = pd.read_parquet(file, columns=['ts'])
        if not df.empty and df['ts'].min() <= unix_ts <= df['ts'].max():
            found_ticker = file.split('/')[-1].replace('dense_', '').replace('.parquet', '')
            break

    print(f"Market Ticker: {found_ticker}")

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader
from typing import List

def audit_favorable_excursions(alpha_model: torch.nn.Module, dense_file: str, raw_df: pd.DataFrame, conf_threshold: float = 0.55, horizon_sec: int = 60) -> None:
    print(f"Auditing Forward Returns for {dense_file}...")
    alpha_model.eval()

    dataset = KalshiDenseDataset([dense_file], seq_len=60, forward_look=30, threshold=0.015, stride=1)
    loader = DataLoader(dataset, batch_size=256, shuffle=False)

    true_positive_ts, directions = [], []

    with torch.no_grad():
        for X_batch, Y_batch, ts_batch in loader:
            logits = alpha_model(X_batch.to(device))
            probs = torch.nn.functional.softmax(logits, dim=1)
            max_probs, preds = torch.max(probs, dim=1)

            for i in range(len(preds)):
                if max_probs[i] >= conf_threshold and preds[i] == Y_batch[i] and preds[i] in [0, 2]:
                    true_positive_ts.append(ts_batch[i].item())
                    directions.append('UP' if preds[i] == 2 else 'DOWN')

    print(f"Found {len(true_positive_ts)} True Positive directional signals. Simulating future paths...")

    if not true_positive_ts:
        return

    mfe_list, mae_list = [], []

    for ts, direction in zip(true_positive_ts, directions):
        # FIXED: Using 'ts' instead of 'local_recv_ts'
        future_df = raw_df[(raw_df['ts'] >= ts) & (raw_df['ts'] <= ts + horizon_sec)]

        if future_df.empty:
            continue

        entry_bid = future_df.iloc[0]['bid_px_1']
        entry_ask = future_df.iloc[0]['ask_px_1']

        if direction == 'UP':
            peak_bid = future_df['bid_px_1'].max()
            low_bid = future_df['bid_px_1'].min()
            mfe_list.append(peak_bid - entry_ask)
            mae_list.append(low_bid - entry_ask)
        else:
            low_ask = future_df['ask_px_1'].min()
            peak_ask = future_df['ask_px_1'].max()
            mfe_list.append(entry_bid - low_ask)
            mae_list.append(entry_bid - peak_ask)

    mfe_series = pd.Series(mfe_list) * 100
    mae_series = pd.Series(mae_list) * 100

    print("\n=== OPTIMAL TAKE-PROFIT AUDIT (MFE) ===")
    print("If we held perfectly, how far did the market move in our favor?")
    print(f"Average Move: +{mfe_series.mean():.2f} cents")
    print(f"50% of trades hit: +{mfe_series.quantile(0.50):.2f} cents")
    print(f"80% of trades hit: +{mfe_series.quantile(0.20):.2f} cents")
    print(f"Absolute Peak (100%): +{mfe_series.max():.2f} cents")

    print("\n=== OPTIMAL STOP-LOSS AUDIT (MAE) ===")
    print("How far did the price dip against us before recovering?")
    print(f"Average Dip: {mae_series.mean():.2f} cents")
    print(f"80% of trades dipped no further than: {mae_series.quantile(0.20):.2f} cents")

# ==========================================
# === EXECUTION SCRIPT ===
# ==========================================

# 1. Grab the first file and figure out its ticker name
first_dense_file = val_files[0]
ticker_name = os.path.basename(first_dense_file).replace('dense_', '').replace('.parquet', '')
print(f"Hunting down raw data for {ticker_name}...")

# 2. Physically locate its raw parquet files
raw_files = []
for root, dirs, files in os.walk('/content/kalshi_extracted'):
    if ticker_name in root:
        for file in files:
            if file.endswith('.parquet'):
                raw_files.append(os.path.join(root, file))

# 3. Build the raw_df using the correct mixed-schema timestamp parsing
temp_raw_df: pd.DataFrame = pd.concat([pd.read_parquet(f) for f in raw_files], ignore_index=True)

# Try numeric first, then fall back to ISO datetime for the failures
parsed_ts: pd.Series = pd.to_numeric(temp_raw_df['ts'], errors='coerce')
iso_mask: pd.Series = parsed_ts.isna()

if iso_mask.any():
    parsed_ts[iso_mask] = pd.to_datetime(temp_raw_df.loc[iso_mask, 'ts'], format='mixed', utc=True).astype('int64') / 1e9

temp_raw_df['ts'] = parsed_ts
temp_raw_df = temp_raw_df.sort_values(by=['ts', 'seq']).reset_index(drop=True)

In [ ]:
import torch
import torch.nn as nn

class ZeroIntelligenceModel(nn.Module):
    """
    A dummy agent for baseline simulation.
    Generates random logits heavily weighted toward FLAT (Class 1).
    """
    def __init__(self):
        super().__init__()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size: int = x.size(0)

        # Base logits to approximate [DOWN: 5%, FLAT: 90%, UP: 5%] before softmax
        # ln(0.05) ≈ -2.99, ln(0.90) ≈ -0.10
        base_logits: torch.Tensor = torch.tensor([[-2.99, -0.10, -2.99]], device=x.device).repeat(batch_size, 1)

        # Inject random noise so it actually triggers trades occasionally when it spikes over the threshold
        noise: torch.Tensor = torch.randn_like(base_logits) * 0.75

        return base_logits + noise

# --- Execution ---
# 1. Instantiate the dummy agent and put it on the same device as your tensors
random_agent = ZeroIntelligenceModel().to(device)
random_agent.eval()

# 2. Feed it directly into your existing simulator
print("Running Zero-Intelligence Baseline...")
df_random_audit: pd.DataFrame = run_master_backtest(
    alpha_model=random_agent,
    val_dense_files=val_files,
    raw_extracted_dir='/content/kalshi_extracted'
)